# 01 多腕バンディット — 「探索」と「活用」のせめぎ合い

## ゴール
- 強化学習の核心である**探索と活用**のジレンマを、最も簡単な問題で体験する。
- **探索なし（貪欲）・ε-greedy・UCB・トンプソン**を比べ、「**賢く探索する**」と何が良くなるかを知る。
- 学習の良し悪しを、**後悔（regret）**という指標で測る。

## 前のNotebookとのつながり
00 では、迷路（状態があり、行動で状態が変わる問題）の基準を作りました。
ここでは、その**最も簡単な形**（状態がなく、行動の結果の報酬だけを見る問題）で、強化学習の核心を調べます。

## 多腕バンディットとは
スロットマシンが10台あり、**台（腕）ごとに、当たりやすさ（平均の報酬）が違います**。どれが良い台かは**分かりません**。
引いて報酬を見ながら、できるだけ**たくさん稼ぐ**のが目的です。

- **活用**: 今のところ最も稼げそうな台を引く。
- **探索**: まだよく知らない台を試す。もしかしたら、もっと良い台かもしれない。

活用ばかりだと、**最初の運で選んだ台に居座って**、もっと良い台を見逃します。探索ばかりだと、良い台が分かっても**引かない**で損をします。
このバランスが、強化学習の中心的な難しさです。

## やること
1. 「少し試しただけでは、最良の腕は分からない」ことを確かめる
2. 探索の量（ε）を変えて比べる
3. 楽観的な初期値、UCB、トンプソン抽出を試す
4. 環境が時間で変わる場合を試す

## キーワード
- **腕**: 選べる行動。**報酬**: 引いた結果の点数。
- **ε-greedy**: 確率εでランダムに腕を選び（探索）、それ以外は今の推定が最大の腕を選ぶ（活用）。ε=0 は**貪欲**（探索なし）。
- **UCB**: 「推定＋不確かさのボーナス」が最大の腕を選ぶ。あまり引いていない腕ほどボーナスが大きい（まだ分からないので試す価値がある）。
- **トンプソン抽出**: 各腕の価値を確率分布で持ち、そこから1つずつ**サンプル**して、最大の腕を選ぶ。分からない腕ほど、たまたま高い値が出て、試される。
- **後悔（regret）**: 「いつも最良の腕を引いていたら得られたはずの報酬」との差の累積。**小さいほど良い**。
- **標準誤差**: 平均のばらつき（誤差）の目安。差がこれより小さいときは、差があるとは言えない。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from bandits import BanditBatch, EpsilonGreedy, UCB, Thompson, run_experiment, summarize
import run_phase1

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

env = BanditBatch(n_runs=1, k=10, seed=3)
print("10本の腕の、真の平均（本当は分からない）:")
print(np.round(env.mu[0], 2), "→ 最良の腕は", int(env.best_arm()[0]), "番")

**何をしているか**: 10本の腕を持つ問題を1つ作ります。真の平均（ここでは見せていますが、**エージェントには分かりません**）が、腕ごとに違います。

**出力の見方**: 腕ごとの平均が、約 −2 から 2 まで散らばっています。1つだけ、いちばん良い腕があります。

In [ ]:
rng = np.random.default_rng(0)
rows = []
for pulls in [1, 3, 10, 30, 100]:
    ok = 0
    for i in range(2000):
        e = BanditBatch(1, 10, seed=int(rng.integers(0, 2**31 - 1)))
        est = np.zeros(10)
        for a in range(10):                      # 各腕を pulls 回ずつ引いて、平均を取る
            est[a] = np.mean([e.pull(np.array([a]))[0] for _ in range(pulls)])
        ok += est.argmax() == e.best_arm()[0]
    rows.append({"各腕を引いた回数": pulls, "最良の腕を当てた割合": ok / 2000})
pd.DataFrame(rows).set_index("各腕を引いた回数")

**何をしているか**: 各腕を**同じ回数ずつ**引いて、平均の報酬が最大の腕を「最良」と判断します。2000個の問題で、それが**本当の最良の腕**だった割合を見ます。

**出力の見方**: 各腕を**1回ずつ**引いただけでは、最良の腕を当てられるのは**43%**だけです。3回ずつで61%、10回ずつで74%、30回ずつで86%、**100回ずつ引いても92%**で、完全には当たりません。報酬には**ノイズ**があるので、少し試しただけでは、運に左右されます。
**引く回数を増やすほど当たりますが**、全部の腕を何度も試す（探索）ほど、良い腕を引く回数（活用）が減ります。これが**探索と活用のジレンマ**です。

In [ ]:
def show(exp, ymax=None):
    methods = run_phase1.EXPERIMENTS[exp]
    steps = run_phase1.EXPERIMENT_STEPS.get(exp, run_phase1.STEPS)
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    rows = []
    for name, (mk, env_kw) in methods.items():
        res = run_experiment(mk, n_runs=run_phase1.N_RUNS, steps=steps, k=run_phase1.K, seed=run_phase1.SEED, **env_kw)
        ax[0].plot(res["reward"], label=name, lw=1.2)
        ax[1].plot(res["optimal"] * 100, label=name, lw=1.2)
        ax[2].plot(res["regret"], label=name, lw=1.2)
        s = summarize(res)
        rows.append(dict(手法=name, 平均報酬_最後=round(s["平均報酬_最後"], 3), 最良の腕の割合_最後=round(s["最良の腕の割合_最後"], 3),
                         累積の後悔=round(s["累積の後悔"], 1), 後悔の標準誤差=round(s["後悔の標準誤差"], 1)))
    ax[0].set(title="平均の報酬", xlabel="何回目に引いたか"); ax[1].set(title="最良の腕を引いた割合 (%)", xlabel="何回目に引いたか")
    ax[2].set(title="累積の後悔（小さいほど良い）", xlabel="何回目に引いたか")
    ax[0].legend(fontsize=8); [a.grid(alpha=0.3) for a in ax]
    fig.suptitle(exp); fig.tight_layout()
    return pd.DataFrame(rows).set_index("手法")

show("A 探索の量（ε）")

**何をしているか**: **探索の量ε**だけを変えて（0・0.01・0.1・0.3）、2000個の問題（各1000回引く）の平均を見ます。

**出力の見方**:
- **貪欲（ε=0）**: 最初は他と同じように上がりますが、すぐ**頭打ち**（平均報酬約1.0）になります。最良の腕を引く割合は**36%**どまりで、横ばい。**最初の運で選んだ腕に居座って**、もっと良い腕を試さないからです。累積の後悔は**501**（誤差±13と大きい）。
- **ε=0.01**: 探索が少ないので、**ゆっくり**上がります。後悔は334。
- **ε=0.1**: 探索と活用のバランスが良く、最も良い。最良の腕の割合は**79%**、後悔は**231**。
- **ε=0.3**: 探索が多すぎて、**最良の腕が分かっても、3割はランダムに引く**ので、頭打ちが低い（平均報酬1.05、最良の腕の割合68%、後悔485）。最良の腕の割合は貪欲より高いのに、**報酬も後悔も貪欲とほぼ同じ**になります。
- つまり、**探索は多すぎても少なすぎても損**で、**中間に最適**があります。

In [ ]:
show("B 楽観的な初期値")

**何をしているか**: 腕の価値の**初期値**を、0 ではなく**5（実際の報酬より、ずっと高い）**にします。探索はしません（貪欲）。

**出力の見方**: 楽観的な初期値の貪欲は、**後悔が501 → 126**に減り、ε=0.1（231）より良くなります。
最初は、どの腕も「5点もらえるはず」と思っているので、**引いてみて報酬が5より低いと、がっかりして別の腕を試します**。結果として、**最初に全部の腕を試す**ことになります（探索が自動的に起きる）。
ただし、この効果は**最初だけ**です。環境が途中で変わる場合には、効きません。

In [ ]:
show("C 賢い探索（UCB・トンプソン）")

**何をしているか**: ε-greedy（ε=0.1）と、**UCB**（c=1, 2）、**トンプソン抽出**を比べます。

**出力の見方**:
- **UCB（c=1）**: 最良の腕の割合**93%**、後悔**59**。ε=0.1（231）の**約4分の1**。
- **トンプソン抽出**: 93%に近い92%、後悔**70**。UCBと同程度に良い。
- **UCB（c=2）**: 探索のボーナスが大きすぎて、後悔150。**cの値で性能が変わる**。
- ε-greedy は、**探索のとき、完全にランダム**に引くので無駄が多い。UCB・トンプソンは、**不確かな腕（まだ分からない腕）だけを**賢く試すので、効率が良い。
- 後悔の**標準誤差**（0.6〜0.8）は、手法間の差（59〜231）よりずっと小さいので、**差は誤差ではありません**。

In [ ]:
show("D 環境が変わる（非定常）")

**何をしているか**: 腕の真の平均が、**時間とともに少しずつランダムに動く**（環境が変わる）問題です。5000回引きます。
ε=0.1 で、**標本平均**（過去のすべての報酬の平均）と、**一定の学習率 α=0.1**（最近の報酬を重く見る）を比べます。

**出力の見方**:
- 標本平均は、**古い報酬も同じ重さで平均する**ので、環境が変わると**古い情報に引きずられ**ます。最良の腕の割合63%、後悔1284。
- 一定の学習率は、**最近の報酬を重視**するので、変化に**追いつけます**。最良の腕の割合79%、後悔1086（誤差±7〜11）。
- 環境が変わらないとき（標準）は、標本平均が正解に収束して良いのですが、**変わる環境では、忘れる**工夫（一定の学習率）が要ります。

In [ ]:
show("E 当たり外れ（ベルヌーイ）")

**何をしているか**: 報酬が「当たり（1）か外れ（0）」の問題で、ε-greedy・UCB・トンプソンを比べます。

**出力の見方**: トンプソン抽出が最も良く、後悔**28**（ε=0.1は68、UCB(c=1)は89）。UCBは、このc=1では、探索が足りず、最良の腕の割合が66%と低い。
**問題の種類（報酬の分布）が変わると、最良の手法・パラメータも変わる**ことが分かります。トンプソンは、報酬の分布に合わせて作る（ここでは、ベータ分布）ので、当たり外れの問題に強い。

## 試してみよう（演習）
1. **探索率を細かく調べる**: 実験Aの `eps` を `[0.02, 0.05, 0.1, 0.2]` にして（`src/run_phase1.py` の `EXPERIMENTS`）、`show` を実行してください。後悔が最小になる `eps` を探します。分かるのは、最適な探索の量は、問題によって決まるということです。
2. **楽観的な初期値を変える**: `q0=5.0` を `1.0` と `20.0` に変えてください。分かるのは、初期値が**低すぎると探索が起きず**、**高すぎると探索しすぎる**ことです。
3. **腕の数を増やす**: `run_phase1.K = 10` を `50` に変えて、貪欲・ε=0.1・UCBを比べてください（`show` を実行する前にセルで `run_phase1.K = 50`）。腕が増えると、探索の費用が増え、賢い探索（UCB・トンプソン）の価値が上がります。
4. **ばらつきを見る**: `n_runs=2000` を `20` に変えて、同じ実験を何度か（`seed` を変えて）実行してください。少数の問題では、手法の順位が入れ替わることがあります。**多くの問題で平均を取る**理由です。

## まとめ
- **探索なし（貪欲）**は、最初の運で選んだ腕に居座り、最良の腕を見逃す（後悔501）。**探索が多すぎる**のも損（ε=0.3: 485）。**中間（ε=0.1: 231）**が良い。
- **賢い探索**（UCB・トンプソン）は、不確かな腕だけを試すので、**後悔が約4分の1**（59〜70）。
- **楽観的な初期値**は、最初に全部の腕を試させる簡単な工夫（後悔126）。ただし最初だけ。
- **環境が変わる**なら、一定の学習率で**古い情報を忘れる**工夫が要る。
- **評価は、多くの問題で平均を取り、標準誤差で差を確かめる**。強化学習は、運に左右されやすい。

次の Phase2 では、**状態がある**問題（迷路）で、**Q学習**を使って、報酬から行動を学びます。ここで学んだ探索の工夫（ε-greedy）が、そのまま使われます。